In [11]:
# 1. Import
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


# 2. Load
df = pd.read_csv('/kaggle/input/datasets/durjoychandrapaul/house-price-bangladesh/house_price_bd.csv')


# 3. Clean Price
df['Price_in_taka'] = df['Price_in_taka'].str.replace('৳', '')
df['Price_in_taka'] = df['Price_in_taka'].str.replace(',', '')
df['Price_in_taka'] = df['Price_in_taka'].astype(float)


# 4. Clean
df = df.dropna()
df = df.drop_duplicates()
df = df.drop('Title', axis=1)


# 5. Remove Outliers
q1 = df['Price_in_taka'].quantile(0.25)
q3 = df['Price_in_taka'].quantile(0.75)
iqr = q3 - q1

df = df[(df['Price_in_taka'] > q1 - 1.5 * iqr) &
        (df['Price_in_taka'] < q3 + 1.5 * iqr)]


# 6. Log Transform
df['Price_in_taka'] = np.log1p(df['Price_in_taka'])


# 7. Features & Target
num_features = ['Bedrooms', 'Bathrooms', 'Floor_area']
cat_features = ['City', 'Location']

X = df[num_features + cat_features]
y = df['Price_in_taka']


# 8. Preprocessing Pipeline
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])


# 9. Model Pipeline
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])


# 10. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 11. Train
pipeline.fit(X_train, y_train)


# 12. Predict
y_pred = pipeline.predict(X_test)


# 13. Evaluate
print("R2 Score:", round(r2_score(y_test, y_pred), 3))


# 14. Save Pipeline
pickle.dump(pipeline, open("app.pkl", "wb"))

R2 Score: 0.849
